### Code for the paper "*A Deep Learning Algorithm Predicts Milk Yield and Production Stage of Dairy Cows utilizing Ultrasound Echotexture 3 Analysis of the Mammary Gland*"

It presents the model construction and the produced results for the ?High Lactation ? No Lactation? binary classification problem.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from tensorflow.keras import models
from tensorflow.keras import layers

Reading the file containing labeling info. The "name" column corresponds to an animal and the "dmy" column corresponds to the animal's milk yield.

In [ ]:
df = pd.read_csv("milk_yield_labels.csv")
print("Length of dataframe containing labeling info: ",len(df))
df.head()

Reading the echotexture variable dataset. Each row contains the echotecture variables retrieved from the ultrasound images associated with the "name" variable.

In [ ]:
df_features = pd.read_csv("echotexture_features.csv") #echotexture_features.csv
print ("Length of dataframe containing features: ",len(df_features))

# Deleting unnecessary column
del df_features['Unnamed: 0'] 
df_features.head()

#### Seven echotecture variables are selected and they are concatenated into a new vector of 56 elements (8 photos * 7 variables for each new sample).

In [ ]:
selected_variable = 'Gradient_Mean_value'
selected_variable2 = 'Homogenity'
selected_variable3 = 'Correlation'
selected_variable4 = 'Percentage_non-zero_Gradients'
selected_variable5 = 'St._Deviation'
selected_variable6 = 'Gradient_Variance'
selected_variable7 = 'Entropy'

features = []
counter = 1
element = []
for index, row in df_features.iterrows():
    if counter % 8 == 0:
        name = row['name'][:-4]
        dmy = df.loc[df['name'] == name]['dmy'].values
        element.append(row[selected_variable])
        element.append(row[selected_variable2])
        element.append(row[selected_variable3])
        element.append(row[selected_variable4])
        element.append(row[selected_variable5])
        element.append(row[selected_variable6])
        element.append(row[selected_variable7])
        element.append(dmy[0])
        element.append(name)
        features.append(element)
        element = []
    else:
        element.append(row[selected_variable])
        element.append(row[selected_variable2])
        element.append(row[selected_variable3])
        element.append(row[selected_variable4])
        element.append(row[selected_variable5])
        element.append(row[selected_variable6])
        element.append(row[selected_variable7])
    counter += 1

# Creating a new dataframe of 56 elements extracted from echotecture variables + 'dmy', 'name'
col = ['A1','xA1','cA1','vA1','bA1','nA1','mA1', 'A2','xA2',\
       'cA2','vA2','bA2','nA2','mA2', 'B1', 'xB1','cB1','vB1','bB1','nB1','mB1', \
        'B2','xB2','cB2','vB2','bB2','nB2','mB2',\
       'C1','xC1','cC1','vC1','bC1','nC1','mC1',\
       'C2','xC2','cC2','vC2','bC2','nC2','mC2', \
        'D1','xD1','cD1','vD1','bD1','nD1','mD1',\
       'D2','xD2','cD2','vD2','bD2','nD2','mD2','dmy','name']

df2 = pd.DataFrame(features,columns = col)
print("length of dataframe: ",len(df2))
print(df2.tail(5))

Some more data cleansing.

In [ ]:
#Remove elements with ''-'' appearing in some values of the dmy.
df2.drop(df2[df2['dmy'] == '-'].index, inplace = True)
df2['dmy'] = df2['dmy'].astype(float)
print("Length of dataframe after dropping rows: ",len(df2))

Label creation by grouping together features with daily milk yield lower than 10 for the No-Lactation class and greater than 45 for the High-Lactation class.

In [ ]:
labels = []
for _,row in df2.iterrows():
    if row['dmy'] >= 0 and row['dmy'] <= 10:
        labels.append(0)
    elif row['dmy'] <= 1000 and row['dmy'] > 45:
        labels.append(1)
    else:
        # All the other cases are of no interest
        labels.append(-1)
        
df2['labels'] = labels
print("Length of dataframe: ", len(df2))

# Drop rows with label value -1
index_names = df2[ df2['labels'] == -1 ].index
df2.drop(index_names , inplace=True)

print("New length of dataframe after dropping classes between No and High Lactation: ", len(df2))
df2.head()

### Model creation - Classes selection - Training - Validation - Testing

In [ ]:
model = models.Sequential()
# Input - Layer
model.add(layers.Dense(50, activation = "relu", input_shape=(56, )))
# Hidden - Layers
model.add(layers.Dropout(0.3, noise_shape=None, seed=None))
model.add(layers.Dense(50, activation = "relu"))
# Output- Layer
model.add(layers.Dense(1, activation = "sigmoid"))
model.summary()
model.compile(
 optimizer = 'sgd',
 loss = "binary_crossentropy",
 metrics = ["accuracy"]
)

all_features = ['A1','xA1','cA1','vA1','bA1','nA1','mA1', 'A2','xA2',\
       'cA2','vA2','bA2','nA2','mA2', 'B1', 'xB1','cB1','vB1','bB1','nB1','mB1', \
        'B2','xB2','cB2','vB2','bB2','nB2','mB2',\
       'C1','xC1','cC1','vC1','bC1','nC1','mC1',\
       'C2','xC2','cC2','vC2','bC2','nC2','mC2', \
        'D1','xD1','cD1','vD1','bD1','nD1','mD1',\
       'D2','xD2','cD2','vD2','bD2','nD2','mD2']

X = df2[all_features].values.tolist()
Y = df2[['labels']].values.tolist()
X = np.array(X)
Y = [y[0] for y in Y]

# Splitting to training, validation and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25,stratify=Y)
X_test, final_x_test, y_test, final_y_test = train_test_split(X_test, y_test, test_size=0.5,stratify=y_test)

# Data transformation and normalization
y_train = np.array(y_train)
y_test = np.array(y_test)
final_y_test= np.array(final_y_test)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_TR = scaler.transform(X_train)
X_test_TR = scaler.transform(X_test)
final_x_test_TR = scaler.transform(final_x_test)

#Train the model
epochs = 50
batch_size = 14
results = model.fit(
    X_train_TR, y_train,
    epochs = epochs,
    batch_size = batch_size,
    validation_data = (X_test_TR, y_test)
)

# Validation set
print("-----Validation set------")
print("Validation Accuracy: ", results.history["val_accuracy"][-1])

pred = model.predict(X_test_TR)
rounded = []
for p in pred:
    rounded.append(np.round(p))

print(classification_report(y_test, rounded))

# Test set
print("-----Unseen test set predictions------")
pred = model.predict(final_x_test_TR)
rounded = []
for p in pred:
    rounded.append(np.round(p))

print(classification_report(final_y_test, rounded))
print("Confusion Matrix of unseen test set: \n", confusion_matrix(final_y_test, rounded))

model_name = 'milk_yield'+"_ep_"+str(epochs)+"_bss_"+str(batch_size) 

# "Loss"
plt.plot(results.history['loss'])
plt.plot(results.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.savefig('loss_valloss_'+model_name+'.png', dpi=600)
plt.show()

#  "Accuracy"
plt.plot(results.history['accuracy'])
plt.plot(results.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.savefig('acc_vallacc_'+model_name+'.png', dpi=600)
plt.show()

Plotting the ROC curve.

In [ ]:
from sklearn import metrics
from sklearn.metrics import roc_curve,roc_auc_score

In [ ]:
def plot_roc_curve(fpr,tpr,model_name): 
    plt.plot(fpr,tpr) 
    plt.axis([0,1,0,1]) 
    plt.xlabel('False Positive Rate') 
    plt.ylabel('True Positive Rate') 
     
    plt.savefig('roc_curve'+model_name+'.png', dpi=600)
    plt.show() 
    
fpr , tpr , thresholds = roc_curve( final_y_test, rounded)#y_unseen_stacked , predictions)
plot_roc_curve (fpr,tpr,model_name) 